# LeetCode 2194 — Cells in a Range on an Excel Sheet bằng Python

Link đề: https://leetcode.com/problems/cells-in-a-range-on-an-excel-sheet/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def cellsInRange(self, s: str) -> List[str]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu cách Excel đặt tên ô: **chữ cái = cột**, **số = hàng**, ví dụ `K1` là cột `K`, hàng `1`.
2. Dùng `ord` và `chr` để đổi qua lại giữa **ký tự và mã số**, từ đó "đếm" các chữ cái liên tiếp.
3. Viết **hai vòng lặp lồng nhau** sinh mọi cặp (cột, hàng) đúng thứ tự đề yêu cầu (cột ngoài, hàng trong).
4. Dùng **list comprehension**, `itertools.product` và lát cắt `string.ascii_uppercase` để viết gọn.
5. Tránh các lỗi kinh điển: đảo thứ tự vòng lặp, `range` thiếu điểm cuối, nhầm `'1'` (ký tự) với `1` (số).

## 2. Đề bài & ví dụ

Một ô trong bảng tính Excel được ký hiệu `<cột><hàng>`: cột là chữ cái in hoa (`A`, `B`, ...), hàng là số nguyên (`1`, `2`, ...). Cho chuỗi `s = "<c1><r1>:<c2><r2>"` mô tả một **vùng hình chữ nhật** từ ô góc trên-trái đến ô góc dưới-phải. Hãy trả về **danh sách mọi ô** trong vùng đó, sắp xếp **tăng dần theo cột trước, rồi theo hàng**.

| Ví dụ | s | Output |
|---|---|---|
| 1 | `"K1:L2"` | `["K1","K2","L1","L2"]` |
| 2 | `"A1:F1"` | `["A1","B1","C1","D1","E1","F1"]` |

- Ví dụ 1: cột `K` có hàng 1, 2; cột `L` có hàng 1, 2. Cột `K` đứng trước nên `K1, K2` đi trước `L1, L2`.
- Ví dụ 2: chỉ có một hàng nên mỗi cột chỉ có một ô.

Constraints: `s.length == 5`; `'A' <= s[0] <= s[3] <= 'Z'`; `'1' <= s[1] <= s[4] <= '9'`; `s[2] == ':'`.

Vì `s` luôn dài đúng 5 ký tự, cột và hàng đều **chỉ một ký tự** — không có cột `AA` hay hàng `10`.

In [1]:
examples = [
    ("K1:L2", ["K1", "K2", "L1", "L2"]),
    ("A1:F1", ["A1", "B1", "C1", "D1", "E1", "F1"]),
]
for s, expected in examples:
    print(s, "->", expected, "| dài", len(s), "| c1 =", s[0], "r1 =", s[1], "c2 =", s[3], "r2 =", s[4])

K1:L2 -> ['K1', 'K2', 'L1', 'L2'] | dài 5 | c1 = K r1 = 1 c2 = L r2 = 2
A1:F1 -> ['A1', 'B1', 'C1', 'D1', 'E1', 'F1'] | dài 5 | c1 = A r1 = 1 c2 = F r2 = 1


## 3. Kiến thức nền

### 3.1. Lấy từng phần của chuỗi bằng chỉ số

Với `s = "K1:L2"`: `s[0] = 'K'` (cột đầu), `s[1] = '1'` (hàng đầu), `s[2] = ':'`, `s[3] = 'L'` (cột cuối), `s[4] = '2'` (hàng cuối). Chú ý `s[1]` là **ký tự** `'1'`, chưa phải số `1`.

In [2]:
s = "K1:L2"
c1, r1, c2, r2 = s[0], s[1], s[3], s[4]
print(c1, r1, c2, r2)
print(type(r1), r1 == "1", r1 == 1)
print(int(r1) + 1)

K 1 L 2
<class 'str'> True False
2


### 3.2. `ord` và `chr`: ký tự ↔ mã số

`ord('A') = 65`, `ord('B') = 66`, ... liên tiếp theo thứ tự bảng chữ cái. `chr(65) = 'A'`. Do đó **"chữ cái tiếp theo" = `chr(ord(c) + 1)`**, và để duyệt từ cột `K` đến `L` ta duyệt mã số từ `ord('K')` đến `ord('L')`.

In [3]:
print(ord("A"), ord("K"), ord("L"), ord("Z"))
print(chr(65), chr(75), chr(ord("K") + 1))
print([chr(code) for code in range(ord("C"), ord("F") + 1)])

65 75 76 90
A K L
['C', 'D', 'E', 'F']


### 3.3. `range` không chứa điểm cuối

`range(a, b)` đi từ `a` đến `b - 1`. Muốn gồm **cả** `b` ta viết `range(a, b + 1)`. Quên `+ 1` là lỗi rất phổ biến khi duyệt "từ ... đến ..." bao gồm hai đầu.

In [4]:
print(list(range(1, 4)))
print(list(range(1, 4 + 1)))
print(list(range(ord("K"), ord("L"))), list(range(ord("K"), ord("L") + 1)))

[1, 2, 3]
[1, 2, 3, 4]
[75] [75, 76]


### 3.4. Hai vòng `for` lồng nhau và thứ tự sinh phần tử

Vòng **ngoài** đổi chậm, vòng **trong** đổi nhanh. Đề muốn "theo cột trước, rồi theo hàng" nghĩa là: với mỗi cột (ngoài), chạy hết các hàng (trong).

In [5]:
for col in "KL":
    for row in "12":
        print(col + row, end=" ")
print()
for row in "12":
    for col in "KL":
        print(col + row, end=" ")
print("  <- đảo vòng: sai thứ tự theo đề")

K1 K2 L1 L2 
K1 L1 K2 L2   <- đảo vòng: sai thứ tự theo đề


### 3.5. List comprehension và `itertools.product`

`[expr for a in A for b in B]` là cách viết gọn cho hai vòng lồng nhau (vòng viết **trước** là vòng ngoài). `itertools.product(A, B)` sinh các cặp `(a, b)` theo đúng thứ tự đó. Cuối cùng `"".join` hoặc phép `+` ghép ký tự.

In [6]:
from itertools import product

print([c + r for c in "KL" for r in "12"])
print(list(product("KL", "12")))
print(["".join(p) for p in product("KL", "12")])

['K1', 'K2', 'L1', 'L2']
[('K', '1'), ('K', '2'), ('L', '1'), ('L', '2')]
['K1', 'K2', 'L1', 'L2']


### 3.6. Hằng `string.ascii_uppercase` và lát cắt

`string.ascii_uppercase` là chuỗi `"ABC...XYZ"`. Dùng `.index(c)` để biết vị trí và lát cắt để lấy một đoạn cột — không cần `ord/chr`.

In [7]:
import string

up = string.ascii_uppercase
print(up)
print(up.index("K"), up.index("L"), up[up.index("K"): up.index("L") + 1])

ABCDEFGHIJKLMNOPQRSTUVWXYZ
10 11 KL


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy vẽ vùng `K1:L2` như một bảng nhỏ:

| | **K** | **L** |
|---|---|---|
| **1** | K1 | L1 |
| **2** | K2 | L2 |

Đề muốn đọc **từng cột từ trên xuống dưới**, rồi sang cột kế tiếp: `K1, K2, L1, L2`. Vậy bài toán thật ra là: "liệt kê các cột từ `c1` đến `c2`, liệt kê các hàng từ `r1` đến `r2`, rồi ghép **mọi cặp** theo thứ tự cột-trước". Số ô tổng cộng là `(c2 - c1 + 1) * (r2 - r1 + 1)`.

In [8]:
s = "K1:L2"
cols = ord(s[3]) - ord(s[0]) + 1
rows = int(s[4]) - int(s[1]) + 1
print("số cột =", cols, "| số hàng =", rows, "| tổng ô =", cols * rows)
s = "A1:F1"
print((ord(s[3]) - ord(s[0]) + 1) * (int(s[4]) - int(s[1]) + 1))

số cột = 2 | số hàng = 2 | tổng ô = 4
6


## 5. Cách 1 — Brute Force / baseline: hai vòng `for` với `ord/chr`

Chuyển cột về mã số, duyệt `range(ord(c1), ord(c2) + 1)`, và hàng về `int`, duyệt `range(int(r1), int(r2) + 1)`. Mỗi cặp ghép lại thành chuỗi.

In [9]:
def cells_manual(s):
    c1, r1, c2, r2 = s[0], int(s[1]), s[3], int(s[4])
    res = []
    for code in range(ord(c1), ord(c2) + 1):
        for row in range(r1, r2 + 1):
            res.append(chr(code) + str(row))
    return res

print(cells_manual("K1:L2"))
print(cells_manual("A1:F1"))

['K1', 'K2', 'L1', 'L2']
['A1', 'B1', 'C1', 'D1', 'E1', 'F1']


In [10]:
from typing import List

class SolutionLoops:
    def cellsInRange(self, s: str) -> List[str]:
        res = []
        for code in range(ord(s[0]), ord(s[3]) + 1):
            for row in range(int(s[1]), int(s[4]) + 1):
                res.append(chr(code) + str(row))
        return res

print(SolutionLoops().cellsInRange("K1:L2"))
print(SolutionLoops().cellsInRange("A1:F1"))

['K1', 'K2', 'L1', 'L2']
['A1', 'B1', 'C1', 'D1', 'E1', 'F1']


**Độ phức tạp:** Gọi `C` số cột, `R` số hàng. Phải sinh `C·R` chuỗi nên **Time `O(C·R)`**, **Space `O(C·R)`** cho kết quả (không tính vào bộ nhớ phụ thì phụ `O(1)`). Vì `C ≤ 26`, `R ≤ 9` nên tối đa `234` ô.

**Nút thắt:** không có — đầu ra bắt buộc phải liệt kê `C·R` phần tử nên không thể nhanh hơn. Ta chỉ cải thiện **độ gọn** và **độ dễ đọc** ở các cách sau.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: đây là **tích Descartes** của hai dãy: `cols × rows`. Quy tắc:

- nếu hai dãy được sinh theo thứ tự tăng dần thì tích theo kiểu "cột ngoài, hàng trong" cho ra **đúng thứ tự đề yêu cầu** mà không cần sắp xếp lại;
- cột và hàng độc lập nhau: ô `(c, r)` tồn tại ⇔ `c1 ≤ c ≤ c2` và `r1 ≤ r ≤ r2`;
- vì ràng buộc `s[0] <= s[3]`, `s[1] <= s[4]` nên không cần xử lý trường hợp góc bị ngược.

Từ đó ta có nhiều cách viết cùng một ý: vòng lặp, list comprehension, `product`, hay lát cắt trên bảng chữ cái.

## 7. Các cách tiếp theo

### Cách 2 — List comprehension một dòng

Gói hai vòng lặp vào một biểu thức. Vòng viết trước (`code`) là vòng ngoài nên đúng thứ tự cột-trước.

In [11]:
from typing import List

class SolutionComp:
    def cellsInRange(self, s: str) -> List[str]:
        return [chr(c) + str(r)
                for c in range(ord(s[0]), ord(s[3]) + 1)
                for r in range(int(s[1]), int(s[4]) + 1)]

print(SolutionComp().cellsInRange("K1:L2"))

['K1', 'K2', 'L1', 'L2']


**Độ phức tạp:** Time `O(C·R)`, Space `O(C·R)` (kết quả). Cùng bậc với Cách 1 nhưng ngắn hơn và nhanh hơn chút do ít thao tác `append`.

### Cách 3 — `itertools.product` trên hai chuỗi

Sinh hai chuỗi con: cột là lát cắt của `ascii_uppercase`, hàng là chuỗi các chữ số từ `s[1]` đến `s[4]`. `product` ghép từng cặp; `"".join` tạo tên ô.

In [12]:
import string
from itertools import product
from typing import List

class SolutionProduct:
    def cellsInRange(self, s: str) -> List[str]:
        up = string.ascii_uppercase
        cols = up[up.index(s[0]): up.index(s[3]) + 1]
        rows = "123456789"[int(s[1]) - 1: int(s[4])]
        return ["".join(p) for p in product(cols, rows)]

print(SolutionProduct().cellsInRange("K1:L2"))
print(SolutionProduct().cellsInRange("A1:F1"))

['K1', 'K2', 'L1', 'L2']
['A1', 'B1', 'C1', 'D1', 'E1', 'F1']


**Độ phức tạp:** Time `O(C·R)`, Space `O(C·R)`. Ưu điểm: không dùng `ord/chr`, chỉ cần nhớ lát cắt. Lưu ý lát cắt hàng: `"123456789"[r1-1 : r2]` lấy các chữ số từ `r1` đến `r2` (vì chỉ số bắt đầu từ 0).

### Cách 4 — Xây cột và hàng thành danh sách trước, ghép sau

Tách bước "chuẩn bị dữ liệu" khỏi bước "ghép". Cách này dễ debug: ta in được `cols` và `rows` để kiểm tra từng phần.

In [13]:
from typing import List

class SolutionTwoLists:
    def cellsInRange(self, s: str) -> List[str]:
        cols = [chr(x) for x in range(ord(s[0]), ord(s[3]) + 1)]
        rows = [str(x) for x in range(int(s[1]), int(s[4]) + 1)]
        res = []
        for c in cols:
            for r in rows:
                res.append(c + r)
        return res

sol = SolutionTwoLists()
print(sol.cellsInRange("K1:L2"))
print(sol.cellsInRange("B3:D4"))

['K1', 'K2', 'L1', 'L2']
['B3', 'B4', 'C3', 'C4', 'D3', 'D4']


**Độ phức tạp:** Time `O(C·R)`, Space `O(C·R + C + R)` (thêm hai danh sách phụ nhỏ).

### Cách 5 (mở rộng) — Sinh theo chỉ số một vòng `for`

Có `C·R` ô, ô thứ `t` (0-indexed) nằm ở cột thứ `t // R` và hàng thứ `t % R` tính từ góc. Dùng `divmod` thay hai vòng lồng nhau. Kỹ thuật này hữu ích khi muốn "trải phẳng" bảng 2 chiều.

In [14]:
from typing import List

class SolutionDivmod:
    def cellsInRange(self, s: str) -> List[str]:
        C = ord(s[3]) - ord(s[0]) + 1
        R = int(s[4]) - int(s[1]) + 1
        res = []
        for t in range(C * R):
            ci, ri = divmod(t, R)
            res.append(chr(ord(s[0]) + ci) + str(int(s[1]) + ri))
        return res

print(SolutionDivmod().cellsInRange("K1:L2"))
print(SolutionDivmod().cellsInRange("A1:F1"))

['K1', 'K2', 'L1', 'L2']
['A1', 'B1', 'C1', 'D1', 'E1', 'F1']


**Độ phức tạp:** Time `O(C·R)`, Space `O(C·R)`. Đúng nhưng khó đọc hơn hai vòng lồng nhau — dùng để hiểu "trải phẳng", không nhất thiết khuyến nghị khi nộp.

## 8. TRACE từng bước

Trace Cách 1 cho `s = "K1:L2"`: `ord('K') = 75`, `ord('L') = 76`, hàng từ 1 đến 2.

| bước | `code` | `chr(code)` | `row` | ô thêm vào | `res` sau bước |
|---|---|---|---|---|---|
| 1 | 75 | K | 1 | K1 | [K1] |
| 2 | 75 | K | 2 | K2 | [K1, K2] |
| 3 | 76 | L | 1 | L1 | [K1, K2, L1] |
| 4 | 76 | L | 2 | L2 | [K1, K2, L1, L2] |

In [15]:
def trace_cells(s):
    lo_c, hi_c = ord(s[0]), ord(s[3])
    lo_r, hi_r = int(s[1]), int(s[4])
    print(f"s={s} | cột {lo_c}..{hi_c} | hàng {lo_r}..{hi_r}")
    res = []
    step = 0
    for code in range(lo_c, hi_c + 1):
        for row in range(lo_r, hi_r + 1):
            step += 1
            cell = chr(code) + str(row)
            res.append(cell)
            print(f"  bước {step}: code={code} ({chr(code)}), row={row} -> {cell}, res={res}")
    return res

trace_cells("K1:L2")
trace_cells("A2:A4")

s=K1:L2 | cột 75..76 | hàng 1..2
  bước 1: code=75 (K), row=1 -> K1, res=['K1']
  bước 2: code=75 (K), row=2 -> K2, res=['K1', 'K2']
  bước 3: code=76 (L), row=1 -> L1, res=['K1', 'K2', 'L1']
  bước 4: code=76 (L), row=2 -> L2, res=['K1', 'K2', 'L1', 'L2']
s=A2:A4 | cột 65..65 | hàng 2..4
  bước 1: code=65 (A), row=2 -> A2, res=['A2']
  bước 2: code=65 (A), row=3 -> A3, res=['A2', 'A3']
  bước 3: code=65 (A), row=4 -> A4, res=['A2', 'A3', 'A4']


['A2', 'A3', 'A4']

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def cellsInRange(self, s: str) -> List[str]:
        return [chr(c) + str(r)
                for c in range(ord(s[0]), ord(s[3]) + 1)
                for r in range(int(s[1]), int(s[4]) + 1)]

print(Solution().cellsInRange("K1:L2"))
print(Solution().cellsInRange("A1:F1"))

['K1', 'K2', 'L1', 'L2']
['A1', 'B1', 'C1', 'D1', 'E1', 'F1']


## 10. Vì sao thuật toán đúng

- **Đủ và đúng ô:** vùng hình chữ nhật gồm đúng các ô `(c, r)` với `c1 ≤ c ≤ c2` và `r1 ≤ r ≤ r2`. Hai vòng lặp duyệt **mọi** `c` và **mọi** `r` (nhờ `range(..., hi + 1)` bao gồm cả điểm cuối) nên không thiếu, không thừa, và mỗi ô sinh đúng một lần.
- **Đúng thứ tự:** vòng ngoài là cột tăng dần, vòng trong là hàng tăng dần. Hai ô khác cột: ô có cột nhỏ hơn được sinh trước (đúng quy tắc cột-trước). Cùng cột: ô có hàng nhỏ hơn đi trước.
- **Đúng tên ô:** `chr(code)` cho đúng chữ cái cột và `str(row)` cho đúng số hàng; vì hàng chỉ một chữ số nên nối chuỗi không gây nhập nhằng.

## 11. Phân tích Time / Space Complexity

Đặt `C = ord(s[3]) - ord(s[0]) + 1` (số cột), `R = int(s[4]) - int(s[1]) + 1` (số hàng).

| Cách | Time | Space (kể cả kết quả) |
|---|---|---|
| 1. Hai vòng for | `O(C·R)` | `O(C·R)` |
| 2. List comprehension | `O(C·R)` | `O(C·R)` |
| 3. `product` | `O(C·R)` | `O(C·R)` |
| 4. Hai danh sách phụ | `O(C·R)` | `O(C·R + C + R)` |
| 5. `divmod` | `O(C·R)` | `O(C·R)` |

Tối đa `26 · 9 = 234` ô nên mọi cách đều tức thời. **Không thể tốt hơn `O(C·R)`** vì chỉ riêng đầu ra đã có `C·R` phần tử.

## 12. Bộ test

Ví dụ đề, edge case (một ô, một cột, một hàng, toàn bộ bảng), và test ngẫu nhiên có seed đối chiếu **tất cả cách**.

In [17]:
import random

solvers = [SolutionLoops(), SolutionComp(), SolutionProduct(), SolutionTwoLists(), SolutionDivmod(), Solution()]

fixed = [
    ("K1:L2", ["K1", "K2", "L1", "L2"]),
    ("A1:F1", ["A1", "B1", "C1", "D1", "E1", "F1"]),
    ("A1:A1", ["A1"]),                                  # đúng một ô
    ("C3:C5", ["C3", "C4", "C5"]),                      # một cột
    ("Y9:Z9", ["Y9", "Z9"]),                            # sát biên bảng chữ cái
    ("B2:C3", ["B2", "B3", "C2", "C3"]),
]
for s, expected in fixed:
    for sol in solvers:
        assert sol.cellsInRange(s) == expected, (type(sol).__name__, s)
print("test cố định: OK")

full = Solution().cellsInRange("A1:Z9")
assert len(full) == 26 * 9 == 234 and full[0] == "A1" and full[-1] == "Z9"
assert len(set(full)) == len(full)

random.seed(2194)
for _ in range(300):
    c1 = random.randint(0, 25); c2 = random.randint(c1, 25)
    r1 = random.randint(1, 9); r2 = random.randint(r1, 9)
    s = chr(65 + c1) + str(r1) + ":" + chr(65 + c2) + str(r2)
    expected = [chr(65 + c) + str(r) for c in range(c1, c2 + 1) for r in range(r1, r2 + 1)]
    for sol in solvers:
        assert sol.cellsInRange(s) == expected, (type(sol).__name__, s)
    assert len(expected) == (c2 - c1 + 1) * (r2 - r1 + 1)
print("300 test ngẫu nhiên: OK")

test cố định: OK
300 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đảo thứ tự hai vòng lặp

Đề yêu cầu cột trước rồi mới tới hàng. Đặt vòng hàng ở ngoài sẽ ra `K1, L1, K2, L2` — sai thứ tự.

In [18]:
def wrong1(s):
    return [chr(c) + str(r)
            for r in range(int(s[1]), int(s[4]) + 1)
            for c in range(ord(s[0]), ord(s[3]) + 1)]

def right1(s):
    return [chr(c) + str(r)
            for c in range(ord(s[0]), ord(s[3]) + 1)
            for r in range(int(s[1]), int(s[4]) + 1)]

print("sai :", wrong1("K1:L2"))
print("đúng:", right1("K1:L2"))

sai : ['K1', 'L1', 'K2', 'L2']
đúng: ['K1', 'K2', 'L1', 'L2']


### Lỗi 2: `range` thiếu điểm cuối

`range(ord('K'), ord('L'))` chỉ cho `K`, bỏ mất cột `L`.

In [19]:
def wrong2(s):
    return [chr(c) + str(r)
            for c in range(ord(s[0]), ord(s[3]))          # thiếu + 1
            for r in range(int(s[1]), int(s[4]))]         # thiếu + 1

def right2(s):
    return [chr(c) + str(r)
            for c in range(ord(s[0]), ord(s[3]) + 1)
            for r in range(int(s[1]), int(s[4]) + 1)]

print("sai :", wrong2("K1:L2"))
print("đúng:", right2("K1:L2"))

sai : ['K1']
đúng: ['K1', 'K2', 'L1', 'L2']


### Lỗi 3: Nhầm ký tự số với số nguyên

`s[1]` là `'1'` (chuỗi). Ghép `chr(c) + r` khi `r` là số sẽ gây `TypeError`; dùng `range('1', '2')` cũng lỗi.

In [20]:
try:
    print("K" + 1)
except TypeError as e:
    print("sai ->", e)
try:
    print(list(range("1", "3")))
except TypeError as e:
    print("sai ->", e)
print("đúng:", "K" + str(1), list(range(int("1"), int("2") + 1)))

sai -> can only concatenate str (not "int") to str
sai -> 'str' object cannot be interpreted as an integer
đúng: K1 [1, 2]


### Lỗi 4: Cộng mã số thay vì dùng `chr`

`ord('K') + 1` là một **số** (`76`), không phải chữ `'L'`. Phải gọi `chr(...)` để quay lại ký tự.

In [21]:
print(ord("K") + 1, type(ord("K") + 1))
print(chr(ord("K") + 1), type(chr(ord("K") + 1)))
print("K" + str(ord("K") + 1), "<- ghép nhầm số")

76 <class 'int'>
L <class 'str'>
K76 <- ghép nhầm số


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Liệt kê mọi tổ hợp của hai (hoặc nhiều) dãy" → **tích Descartes**: hai vòng lồng, `product` hoặc list comprehension hai `for`.
- "Duyệt chữ cái từ X đến Y" → `range(ord(X), ord(Y) + 1)` rồi `chr`, hoặc lát cắt `ascii_uppercase`.
- Thứ tự đầu ra bị ràng buộc → **vòng ngoài = khóa sắp xếp chính**. Đừng sinh rồi `sort` nếu có thể sinh đúng thứ tự ngay.
- Đọc kỹ constraints: `s.length == 5` cho phép dùng chỉ số cố định `s[0], s[1], s[3], s[4]`, không cần `split`/`parse` phức tạp.
- Trên LeetCode, danh sách kết quả không cần trả về đúng kiểu `tuple`/`generator`; phải là `List[str]`.

## 15. Biến thể của bài

1. Cột và hàng có thể nhiều ký tự (`AA10:AC12`): phải đổi tên cột sang số hệ 26 (xem LeetCode 168 Excel Sheet Column Title và 171 Excel Sheet Column Number).
2. Trả về các ô theo thứ tự **hàng trước, cột sau** (chỉ đảo hai vòng).
3. Đếm số ô thay vì liệt kê: công thức `(c2 - c1 + 1) * (r2 - r1 + 1)` chạy `O(1)`.
4. Chỉ trả về các ô trên **đường chéo** hoặc **viền** của vùng.
5. Cho hai vùng chữ nhật, tìm các ô **giao nhau** (tương giao hai khoảng cột và hai khoảng hàng).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng for | `ord/chr` + `append` | `O(C·R)` | `O(C·R)` | mới học, dễ debug |
| 2. List comprehension | gói hai vòng thành một biểu thức | `O(C·R)` | `O(C·R)` | **khuyến nghị khi nộp** |
| 3. `product` | tích Descartes của cột và hàng | `O(C·R)` | `O(C·R)` | khi có nhiều dãy hơn hai |
| 4. Hai danh sách phụ | chuẩn bị `cols`, `rows` rồi ghép | `O(C·R)` | `O(C·R + C + R)` | muốn in từng phần kiểm tra |
| 5. `divmod` | trải phẳng bảng 2D thành 1 chỉ số | `O(C·R)` | `O(C·R)` | luyện kỹ thuật chỉ số, bài lưới lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print([chr(c) for c in range(ord("B"), ord("E"))])` in ra gì? Chú ý điểm cuối.

**Bài 2 (tính tay).** Với `s = "B2:D3"`, liệt kê đủ các ô theo thứ tự đề yêu cầu và cho biết có bao nhiêu ô.

**Bài 3 (điền TODO).** Hoàn thành hàm trả về các ô theo thứ tự **hàng trước, cột sau**.

In [22]:
def cells_row_first(s):
    # TODO: với mỗi hàng (vòng ngoài), duyệt các cột (vòng trong)
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `count_cells(s)` trả về **số ô** trong vùng mà **không** sinh danh sách.

In [23]:
def count_cells(s):
    # TODO: tự viết, độ phức tạp O(1)
    pass

**Bài 5 (mini challenge).** Viết `border_cells(s)` trả về chỉ các ô nằm trên **viền** của vùng (hàng đầu, hàng cuối, cột đầu, cột cuối), theo thứ tự cột-trước. Ví dụ `"A1:C3"` → `["A1","A2","A3","B1","B3","C1","C2","C3"]`.

In [24]:
def border_cells(s):
    # TODO: sinh mọi ô rồi lọc những ô có cột là đầu/cuối hoặc hàng là đầu/cuối
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `range(ord("B"), ord("E"))` đi tới `ord("D")` rồi dừng. Bài 2: có 3 cột (B, C, D) và 2 hàng (2, 3).
- **Mức 2 (vừa):** Bài 3 chỉ cần đổi vòng ngoài/vòng trong. Bài 4: `(c2 - c1 + 1) * (r2 - r1 + 1)` với `c = ord(...)`, `r = int(...)`.
- **Mức 3 (sâu):** Bài 5: trong vòng lặp cột/hàng, giữ ô khi `col in (c1, c2) or row in (r1, r2)`. Kiểm tra bằng đếm: viền có `2*(C+R)-4` ô khi `C, R ≥ 2`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1 và 2
assert [chr(c) for c in range(ord("B"), ord("E"))] == ["B", "C", "D"]
assert Solution().cellsInRange("B2:D3") == ["B2", "B3", "C2", "C3", "D2", "D3"]

# Bài 3
def cells_row_first(s):
    return [chr(c) + str(r)
            for r in range(int(s[1]), int(s[4]) + 1)
            for c in range(ord(s[0]), ord(s[3]) + 1)]

# Bài 4
def count_cells(s):
    return (ord(s[3]) - ord(s[0]) + 1) * (int(s[4]) - int(s[1]) + 1)

# Bài 5
def border_cells(s):
    c1, c2, r1, r2 = s[0], s[3], int(s[1]), int(s[4])
    return [chr(c) + str(r)
            for c in range(ord(c1), ord(c2) + 1)
            for r in range(r1, r2 + 1)
            if chr(c) in (c1, c2) or r in (r1, r2)]

assert cells_row_first("K1:L2") == ["K1", "L1", "K2", "L2"]
assert count_cells("K1:L2") == 4 and count_cells("A1:Z9") == 234 and count_cells("A1:A1") == 1
assert border_cells("A1:C3") == ["A1", "A2", "A3", "B1", "B3", "C1", "C2", "C3"]
assert len(border_cells("A1:E4")) == 2 * (5 + 4) - 4
assert len(border_cells("A1:A3")) == 3
print(border_cells("A1:C3"))
print("tất cả bài tập OK")

['A1', 'A2', 'A3', 'B1', 'B3', 'C1', 'C2', 'C3']
tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tích Descartes cột × hàng, cột ở vòng ngoài)

```python
class Solution:
    def cellsInRange(self, s: str) -> List[str]:
        return [chr(c) + str(r)
                for c in range(ord(s[0]), ord(s[3]) + 1)
                for r in range(int(s[1]), int(s[4]) + 1)]
```

**Checklist**

- [ ] Vòng ngoài là **cột**, vòng trong là **hàng** (đúng thứ tự đề)?
- [ ] Cả hai `range` đều có `+ 1` ở điểm cuối?
- [ ] Hàng được đổi bằng `int(...)` khi cần `range`, và `str(...)` khi ghép tên ô?
- [ ] Dùng `chr(...)` để đổi mã số về chữ cái?
- [ ] Đã thử vùng một ô (`A1:A1`), một cột, một hàng, và `A1:Z9` (234 ô)?
- [ ] Signature đúng `cellsInRange(self, s: str) -> List[str]` và có `from typing import List`?